# Dataset creation — step-by-step workflow

Every cell runs **one** step. Run them in order, or skip the ones you do not need.
Each step writes into:

```
output/<ISO3>/<YEAR>/
├── country/                         INFORM
└── <EVENTTYPE>_<EVENTID>/
    ├── gdacs/  montandon/  reliefweb/
    └── episodes/<EP>/areas/<LABEL>/<RES>m/
```

Every table has the columns `source, time, time_type, spatial_level, spatial_resolution_m, crs`.

## 0 — Event to process

In [49]:
EVENTTYPE = "EQ"
EVENTID   = 1548377
COUNTRY = "VEN"
YEAR = 2026

## 1 — GDACS: event data (gives ISO3, year → event folder)

In [50]:
from sources import gdacs

ev        = gdacs.get_event_data(EVENTTYPE, EVENTID)        # saved in <event_dir>/gdacs/eventdata.json
event_dir = gdacs.event_dir_from_eventdata(ev)              # output/COL/2026/EQ_1557236
gd        = event_dir / "gdacs"     
event_dir

[GDACS] saved output\VEN\2026\EQ_1548377\gdacs\eventdata.json


WindowsPath('output/VEN/2026/EQ_1548377')

In [51]:
ids = gdacs.episode_ids(gd / "eventdata.json")           # main episode + all episodes
gdacs.event_table(gd / "eventdata.json")

[GDACS] EQ 1548377 - Earthquake in Venezuela
    alert=Red  country=Venezuela  date=2026-06-24T22:05:12
    main episode        : 1714328
    episodes found (2): [1714321, 1714328]
    geometry url        : https://www.gdacs.org/gdacsapi/api/polygons/getgeometry?eventtype=EQ&eventid=1548377&episodeid=1714328


,source,time,time_type,spatial_level,spatial_resolution_m,crs,eventtype,eventid,name,iso3,country,affected_iso3,alertlevel,alertscore,fromdate,todate,episodeid
0,gdacs,2026-06-24T22:05:12,event_start,event,None,None,EQ,1548377,Earthquake in Venezuela,VEN,Venezuela,VEN,Red,3,2026-06-24T22:05:12,2026-06-24T22:05:12,1714328


## 2 — GDACS: card, EM-DAT, news

In [52]:
gdacs.get_event_card(EVENTTYPE, EVENTID, event_dir)
gdacs.summarytable(gd / "adaptivecard.json")

[GDACS] saved output\VEN\2026\EQ_1548377\gdacs\adaptivecard.json


{'Countries': 'Venezuela',
 'Magnitude': '7.5',
 'Depth': '10 Km',
 'Lat/Lon': '10.453 , -68.5139',
 'Event Date': '24 Jun 2026 22:05 UTC',
 'Local Time': '24 Jun 2026 18:05',
 'Exposed population': '6.8 million (in MMI>=VII) - About 68000 people  in MMI>=IX',
 'Maximun Wave Height estimated': '0.28 m'}

In [53]:
gdacs.get_emdat(EVENTTYPE, EVENTID, event_dir)
emdat_df = gdacs.emdat_table(gd / "emdat.json")              # deaths, injured, affected, damage ...
emdat_df

[GDACS] saved output\VEN\2026\EQ_1548377\gdacs\emdat.json


,source,time,time_type,spatial_level,spatial_resolution_m,crs,disasternumber,historical,classificationkey,groupname,...,totalaffected,reconstrdam,reconstrdamadj,insurdam,insurdamadj,totaldam,totaldamadj,cpi,fromDate,toDate
0,emdat,2026-06-24T00:00:00,event_start,event,None,None,2026-0395-VEN,False,nat-geo-ear-gro,Natural,...,9764647,None,None,None,None,30000000.0,None,None,2026-06-24T00:00:00,2026-06-24T23:59:59


In [54]:
gdacs.get_emm_news(EVENTTYPE, EVENTID, event_dir, limit=50)
news_df = gdacs.news_table(gd / "emm_news.json")
news_df.head()

[GDACS] saved output\VEN\2026\EQ_1548377\gdacs\emm_news.json


,source,time,time_type,spatial_level,spatial_resolution_m,crs,emmid,news_source,link,title,pubdate
0,emm_news,2026-06-28T22:00:00,pubdate,event,None,None,huffingtonpost-es-es-76c9d3c0e9d2747c9adf7086a...,huffingtonpost-es-es,https://www.huffingtonpost.es/virales/un-divul...,Un divulgador lanza una advertencia sobre el M...,2026-06-28T22:00:00
1,emm_news,2026-06-28T21:59:00,pubdate,event,None,None,unicef-c3e0220ac8ab297cd671ae72e5a12287,unicef,https://www.unicef.org/press-releases/unicef-6...,"UNICEF: 680,000 children estimated to be in ne...",2026-06-28T21:59:00
2,emm_news,2026-06-28T21:33:00,pubdate,event,None,None,ilnordest-f3e85f94fae2baf6795f0b01417b6ecb,ilnordest,https://www.ilnordest.it/italia-ed-esteri/vene...,"Venezuela, uomo e figlio adolescente salvati q...",2026-06-28T21:33:00
3,emm_news,2026-06-28T21:32:00,pubdate,event,None,None,scmp-f7c577d066630293715969d3646dba64,scmp,https://www.scmp.com/opinion/hong-kong-opinion...,"To tackle elderly poverty crisis, Hong Kong ne...",2026-06-28T21:32:00
4,emm_news,2026-06-28T21:31:00,pubdate,event,None,None,critica-1dc6481b1041671cec1a6e220e3e07d0,critica,https://www.critica.com.pa/deportes/presidente...,Presidente de Corea del Sur ordena investigar ...,2026-06-28T21:31:00


## 3 — GDACS: episodes → time + geometry → one file per affected area

In [55]:
from sources.utils import paths

for ep in ids["episodes"]:
    gdacs.get_episode_data(EVENTTYPE, EVENTID, ep, event_dir)   # time of the episode
    gdacs.get_geometry(EVENTTYPE, EVENTID, ep, event_dir)       # polygons of the episode

[GDACS] saved output\VEN\2026\EQ_1548377\episodes\1714321\episodedata.json
[GDACS] saved output\VEN\2026\EQ_1548377\episodes\1714321\geometry.json
[GDACS] saved output\VEN\2026\EQ_1548377\episodes\1714328\episodedata.json
[GDACS] saved output\VEN\2026\EQ_1548377\episodes\1714328\geometry.json


In [56]:
ids

{'eventtype': 'EQ',
 'eventid': 1548377,
 'episodeid': 1714328,
 'episodes': [1714321, 1714328],
 'geometry_url': 'https://www.gdacs.org/gdacsapi/api/polygons/getgeometry?eventtype=EQ&eventid=1548377&episodeid=1714328'}

In [57]:
for ep in ids["episodes"]:
    ep_dir = paths.episode_dir(event_dir, ep)
    print(ep, gdacs.episode_info(ep_dir / "episodedata.json")["time"])
    gdacs.explode_geometry(ep_dir / "geometry.json")

1714321 2026-06-24T22:05:12
[GDACS] output\VEN\2026\EQ_1548377\episodes\1714321\geometry.json -> output\VEN\2026\EQ_1548377\episodes\1714321\areas  (13 areas)
    - Centroid
    - 100km
    - Intensity_3.5
    - Intensity_4
    - Intensity_4.5
    - Intensity_5
    - Intensity_5.5
    - Intensity_6
    - Intensity_6.5
    - Intensity_7
    - Intensity_7.5
    - Intensity_8
    - Intensity_8.5
1714328 2026-06-24T22:05:12
[GDACS] output\VEN\2026\EQ_1548377\episodes\1714328\geometry.json -> output\VEN\2026\EQ_1548377\episodes\1714328\areas  (13 areas)
    - Centroid
    - 100km
    - Intensity_3.5
    - Intensity_4
    - Intensity_4.5
    - Intensity_5
    - Intensity_5.5
    - Intensity_6
    - Intensity_6.5
    - Intensity_7
    - Intensity_7.5
    - Intensity_8
    - Intensity_8.5


Choose one episode and one area to work on in the next steps.

In [58]:
EPISODE = ids["episodes"][0]
ep_dir  = paths.episode_dir(event_dir, EPISODE)
areas   = gdacs.list_areas(ep_dir)          # {label: area.geojson}
list(areas)

['100km',
 'Centroid',
 'Intensity_3.5',
 'Intensity_4',
 'Intensity_4.5',
 'Intensity_5',
 'Intensity_5.5',
 'Intensity_6',
 'Intensity_6.5',
 'Intensity_7',
 'Intensity_7.5',
 'Intensity_8',
 'Intensity_8.5']

In [59]:
AREA      = "Intensity_4.5"
area_file = areas[AREA]
area_dir  = area_file.parent
area_dir

WindowsPath('output/VEN/2026/EQ_1548377/episodes/1714321/areas/Intensity_4.5')

In [60]:
from sources.gdacs.extract_variables import build_variables, save_variables

build_variables(event_dir)
save_variables(event_dir)


[GDACS] saved output\VEN\2026\EQ_1548377\variables.csv  (476 rows, 2 episodes, 5 levels)


WindowsPath('output/VEN/2026/EQ_1548377/variables.csv')

## 4 — GHSL: population (100 m) and settlement model (1000 m) per pixel

In [61]:
from sources import ghsl
from sources.utils.geo import read_features, dissolve_features, to_raster_crs, save_check

YEAR = paths.parse_event_dir(event_dir)["year"]
smod_tif, pop_tif = ghsl.get_smod_and_pop(YEAR)            # release_root=... to change the GHSL path

[GHSL] SMOD: E2025, 1000 m, V2-0 -> G:\GHSL_restricted_data\GHSL_PRODUCTS\RELEASE\GHSL_R2023\2_DATA\GHS_SMOD_GLOBE_R2023A\GHS_SMOD_E2025_GLOBE_R2023A_54009_1000\V2-0\GHS_SMOD_E2025_GLOBE_R2023A_54009_1000_V2_0.tif
[GHSL] POP: E2025, 100 m, V1-0 -> G:\GHSL_restricted_data\GHSL_PRODUCTS\RELEASE\GHSL_R2023\2_DATA\GHS_POP_GLOBE_R2023A\GHS_POP_E2025_GLOBE_R2023A_54009_100\V1-0\GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0.tif


In [62]:
gdf = read_features(area_file)
gdf = dissolve_features(gdf)                  # optional: avoids double counting
gdf = to_raster_crs(gdf, pop_tif)             # SMOD and POP share the same CRS

pop_cells,  pop_area  = ghsl.extract_pop(gdf, pop_tif, area_dir)     # -> area_dir/100m/
smod_cells, smod_area = ghsl.extract_smod(gdf, smod_tif, area_dir)   # -> area_dir/1000m/
pop_area

[utils] read_features: loaded 1 feature(s), CRS EPSG:4326
[utils] dissolve_features: 1 feature(s) -> 1
[utils] to_raster_crs: reprojected 1 feature(s) to ESRI:54009
[GHSL] POP: extracting over 1 feature(s) from GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0.tif ...
[GHSL] POP: done — 5944731 cell(s) over 1 feature(s), total population 2,550,099
[GHSL] saved output\VEN\2026\EQ_1548377\episodes\1714321\areas\Intensity_4.5\100m\ghsl_pop_E2025_pixels.parquet
[GHSL] saved output\VEN\2026\EQ_1548377\episodes\1714321\areas\Intensity_4.5\100m\ghsl_pop_E2025_area.parquet
[GHSL] SMOD: extracting over 1 feature(s) from GHS_SMOD_E2025_GLOBE_R2023A_54009_1000_V2_0.tif ...
[GHSL] SMOD: done — 86150 cell(s) over 1 feature(s)
[GHSL] saved output\VEN\2026\EQ_1548377\episodes\1714321\areas\Intensity_4.5\1000m\ghsl_smod_E2025_pixels.parquet
[GHSL] saved output\VEN\2026\EQ_1548377\episodes\1714321\areas\Intensity_4.5\1000m\ghsl_smod_E2025_area.parquet


,source,time,time_type,spatial_level,spatial_resolution_m,crs,feature_id,pop_sum,cells_covered
0,ghsl_pop,2025,epoch,area,100.0,ESRI:54009,0,2.550099e+06,5.851425e+06


Optional: QA GeoPackage to open in QGIS.

In [ ]:
# save_check(pop_cells, pop_tif, gdf, "POP", YEAR, out_dir=area_dir / "100m")

## 5 — Sentinel-2: N acquisitions before / on / after the event (10 m)

In [ ]:
# from sources import sentinel2

# EVENT_DATE = gdacs.event_info(gd / "eventdata.json")["fromdate"][:10]

# catalog  = sentinel2.open_catalog()
# aoi      = sentinel2.aoi_from_geojson(area_file)
# items    = sentinel2.search_items(catalog, aoi, EVENT_DATE, span_days=60)
# selected = sentinel2.select_items_per_tile(items, EVENT_DATE, n=2)

In [ ]:
# s2_index = sentinel2.download_clipped(selected, aoi, EVENT_DATE, area_dir)   # -> area_dir/10m/sentinel2/
# s2_index

In [ ]:
# pixel values of ONE downloaded image (one row per 10 m pixel)
# s2_pixels = sentinel2.extract_s2_pixels(read_features(area_file), s2_index["path"].iloc[0], area_dir)
# s2_pixels.head()

## 6 — ReliefWeb (from the GLIDE number in EM-DAT)

In [ ]:
# from sources import reliefweb

# GLIDE = gdacs.glide(gd / "emdat.json")["glide"]       # e.g. EQ-2026-000146
# res   = reliefweb.scrape_reliefweb(GLIDE, event_dir)
# reliefweb.reports_table(event_dir / "reliefweb" / "reliefweb.json").head()

[ReliefWeb] disaster 52573 -> 31 reports


KeyboardInterrupt: 

In [63]:
from sources import reliefweb

reliefweb.scrape_reliefweb(None, r"C:\Users\decosiv\Desktop\codes\dataset_creation_pt2\output\VEN\2026\EQ_1548377", days_after=50)
reliefweb.reports_table(event_dir / "reliefweb" / "reliefweb.json").head()

[ReliefWeb] ======================================================================
[ReliefWeb] EVENT EQ_1548377  Earthquake in Venezuela
    type / country    : EQ / VEN
    start (UTC)       : 2026-06-24T22:05:12+00:00
    magnitude/severity: 7.5
    GLIDE candidates  : ['EQ-2026-000093', 'US6000T7ZC']
    read from         : C:\Users\decosiv\Desktop\codes\dataset_creation_pt2\output\VEN\2026\EQ_1548377\variables.csv
[ReliefWeb] GLIDE EQ-2026-000093: 1 disaster(s) found
    candidate disaster 52608: Venezuela: Earthquakes - Jun 2026
      https://reliefweb.int/taxonomy/term/52608
      [ OK ] GLIDE        expected: EQ-2026-000093               found: EQ-2026-000093-VEN
      [ OK ] hazard type  expected: EQ (EQ)                      found: Earthquake
      [ OK ] country      expected: VEN                          found: VEN
      [ OK ] date         expected: 2026-06-24                   found: 2026-06-24 (0.9 days apart)
      [ ?? ] name         expected: Earthquake in Venezuela   

,source,time,time_type,spatial_level,spatial_resolution_m,crs,reliefweb_id,title,url,report_sources,...,days_after_event,countries,mentions_event_country,other_disasters,format,n_files,pdf_paths,body_text,glide,verdict
0,reliefweb,2026-08-13T00:00:00+00:00,report_date,event,None,None,4225607,Envíos aéreos de ayuda humanitaria a Venezuela...,https://reliefweb.int/node/4225607,Logistics Cluster|World Food Programme,...,49.08,VEN,True,,Infographic,1,C:\Users\decosiv\Desktop\codes\dataset_creatio...,,EQ-2026-000093,MATCH
1,reliefweb,2026-08-12T00:00:00+00:00,report_date,event,None,None,4225455,Venezuela | Earthquake - Operation Update #2 (...,https://reliefweb.int/node/4225455,International Federation of Red Cross and Red ...,...,48.08,VEN,True,,Situation Report,1,C:\Users\decosiv\Desktop\codes\dataset_creatio...,**A. SITUATION ANALYSIS** \n**Description of ...,EQ-2026-000093,MATCH
2,reliefweb,2026-08-12T00:00:00+00:00,report_date,event,None,None,4225432,Venezuela Earthquakes Situation Report #6 (Aug...,https://reliefweb.int/node/4225432,International Medical Corps,...,48.08,VEN,True,,Situation Report,1,C:\Users\decosiv\Desktop\codes\dataset_creatio...,"**FAST FACTS**\n\n- On June 24, two earthquake...",EQ-2026-000093,MATCH
3,reliefweb,2026-08-12T00:00:00+00:00,report_date,event,None,None,4225425,Situation Report No.9: Earthquakes in Venezuel...,https://reliefweb.int/node/4225425,Pan American Health Organization,...,48.08,VEN,True,,Situation Report,1,C:\Users\decosiv\Desktop\codes\dataset_creatio...,**HIGHLIGHTS**\n\n• Seven weeks after the two ...,EQ-2026-000093,MATCH
4,reliefweb,2026-08-10T00:00:00+00:00,report_date,event,None,None,4225314,AMDA Emergency Relief #6: Earthquakes in Venez...,https://reliefweb.int/node/4225314,Association of Medical Doctors of Asia,...,46.08,VEN,True,,News and Press Release,1,C:\Users\decosiv\Desktop\codes\dataset_creatio...,"On 9 August, AMDA dispatched a Japanese nurse ...",EQ-2026-000093,MATCH


In [69]:
from sources.reliefweb.extract import extract_texts
from sources.reliefweb.embed import embed_texts, load_model, load_embeddings

# extract_texts(event_dir)                  # -> reliefweb/texts.json
model = load_model()                      # BAAI/bge-m3, caricato una volta sola
embed_texts(event_dir, model=model)       # -> reliefweb/embeddings.npz + chunks.json
emb, meta = load_embeddings(event_dir)    # meta["chunks"][i] corrisponde a emb[i]


c:\Users\decosiv\Desktop\codes\dataset_creation_pt2\env_displaced\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Embed] loading model BAAI/bge-m3


c:\Users\decosiv\Desktop\codes\dataset_creation_pt2\env_displaced\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\decosiv\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\decosiv\Desktop\codes\dataset_creation_pt2\env_displaced\Lib\site-packages\huggi

[Embed] model loaded | max_seq_length 8192
[Embed] ======================================================================
[Embed] EQ_1548377  GLIDE EQ-2026-000093  (490 documents, sentences=3, overlap=1)
  [4225607] Envíos aéreos de ayuda humanitaria a Venezuela (13 agosto 20 -> 2 chunks
  [4225455] Venezuela | Earthquake - Operation Update #2 (MDRVE015)      -> 149 chunks
  [4225432] Venezuela Earthquakes Situation Report #6 (August 12, 2026)  -> 12 chunks
  [4225425] Situation Report No.9: Earthquakes in Venezuela 2026 (M7.2 a -> 80 chunks
  [4225314] AMDA Emergency Relief #6: Earthquakes in Venezuela - 10 Augu -> 13 chunks
  [4225278] UNFPA Situation Report: Earthquakes in Venezuela (24 June –  -> 22 chunks
  [4225266] Venezuela (República Bolivariana de): Respuesta a Terremotos -> 23 chunks
  [4225265] Venezuela (Bolivarian Republic of): Earthquake Response Situ -> 24 chunks
  [4225089] WFP Venezuela Earthquakes Response Situation Report #08, 10  -> 15 chunks
  [4225077] Venezuela 

Encoding: 100%|██████████| 259/259 [3:04:20<00:00, 42.70s/batch]   


[Embed] saved output\VEN\2026\EQ_1548377\reliefweb\embeddings.npz  shape (8277, 1024)  (31.4 MB)
[Embed] saved output\VEN\2026\EQ_1548377\reliefweb\chunks.json
[Embed] ======================================================================


In [ ]:
gdacs.glide(gd / "emdat.json")

{'glide': None, 'all': []}

## 7 — Montandon: events and impact figures

In [44]:
from sources import montandon

client = montandon.connect()                          # token from MONTANDON_API_TOKEN or prompt
event = montandon.find_gdacs_event(client, EVENTID, EVENTTYPE)[0]

[Montandon] connected to https://montandon-eoapi-stage.ifrc.org/stac (Montandon STAC API)
[Montandon] found 1 item(s) for GDACS event 1515838:
  gdacs-event-1515838
     date    : 2025-12-19T05:39:10Z
     country : ['AFG']
     hazards : ['GH0101', 'EQ', 'nat-geo-ear-gro']
     corr_id : 20251219-AFG-GH0101-1678118-GCDB


In [45]:
related = montandon.find_related_events(client, event, days_before=10, days_after=10)

[Montandon] 14 event, 9 hazard, 10 impact collections
[Montandon] 3 related event(s) for gdacs-event-1515838 (AFG, ['EQ', 'GH0101', 'nat-geo-ear-gro'], ±10/10 days):
  gdacs-events         gdacs-event-1515838                           Δt=0.0h  dist=0.0km  corr=20251219-AFG-GH0101-1678118-GCDB
  pdc-events           pdc-event-da5bfb0f-a643-4949-ac17-0ecbf8f1e83e-386376-1766123931746 Δt=0.0h  dist=0.0km  corr=20251219-AFG-GH0101-1-GCDB
  pdc-events           pdc-event-da5bfb0f-a643-4949-ac17-0ecbf8f1e83e-386376-1766123983066 Δt=0.0h  dist=0.0km  corr=20251219-AFG-GH0101-1-GCDB


In [46]:
imp_corr   = montandon.impacts_by_corr_id(client, related)              # exact, per source
imp_window = montandon.impacts_by_window(client, event, days_after=10)   # dynamic, to check

impacts_df = montandon.impacts_table(imp_corr, imp_window, event=event)
impacts_df[["collection", "category", "type", "value", "unit", "match", "days_from_event"]]

[Montandon] 14 event, 9 hazard, 10 impact collections
[Montandon] 38 impact(s) from 2 corr_id(s)
  pdc-impacts: 38
[Montandon] 14 event, 9 hazard, 10 impact collections
[Montandon] 38 impact(s) in window 2025-12-18T05:39:10Z -> 2025-12-29T05:39:10Z (AFG, ['EQ', 'GH0101', 'nat-geo-ear-gro'])
  pdc-impacts: 38
[Montandon] impacts table: 38 rows ({'both': 38})


,collection,category,type,value,unit,match,days_from_event
0,pdc-impacts,adult_20_24,affected_total,2.620440e+05,None,both,0.0
1,pdc-impacts,adult_20_24,affected_total,2.620440e+05,None,both,0.0
2,pdc-impacts,adult_25_29,affected_total,2.179350e+05,None,both,0.0
3,pdc-impacts,adult_25_29,affected_total,2.179350e+05,None,both,0.0
4,pdc-impacts,adult_30_34,affected_total,1.696500e+05,None,both,0.0
5,pdc-impacts,adult_30_34,affected_total,1.696500e+05,None,both,0.0
6,pdc-impacts,adult_35_39,affected_total,1.294560e+05,None,both,0.0
7,pdc-impacts,adult_35_39,affected_total,1.294560e+05,None,both,0.0
8,pdc-impacts,adult_40_44,affected_total,1.038780e+05,None,both,0.0
9,pdc-impacts,adult_40_44,affected_total,1.038780e+05,None,both,0.0


In [ ]:
ISO3 = paths.parse_event_dir(event_dir)["iso3"]
EVENT_START, EVENT_END   = "2026-08-05T00:00:00Z", "2026-08-20T23:59:59Z"
IMPACT_START, IMPACT_END = "2026-08-05T00:00:00Z", "2026-08-30T00:00:00Z"   # wider: reports come later
HAZARD_CODES = ["nat-geo-ear-gro", "GH0101", "GH0002", "EQ"]

# events  = montandon.search_events(client, ISO3, EVENT_START, EVENT_END)
event = montandon.find_gdacs_event(client, EVENTID, EVENTTYPE)
impacts = montandon.search_impacts(client, event, ISO3, HAZARD_CODES, IMPACT_START, IMPACT_END)
montandon.impacts_table(impacts)

NameError: name 'paths' is not defined

In [ ]:
montandon.impacts_table(impacts)

In [ ]:
montandon.save_montandon(event_dir, events, impacts, query={
    "country_code": ISO3, "hazard_codes": HAZARD_CODES,
    "event_date_range": [EVENT_START, EVENT_END],
    "impact_date_range": [IMPACT_START, IMPACT_END],
})

## 8 — INFORM: country-level indicators

In [ ]:
from sources import inform
ISO3 = paths.parse_event_dir(event_dir)["iso3"] 
inform.list_available_years()
inform_df = inform.build_country_table(ISO3, YEAR)    # -> output/<ISO3>/<YEAR>/country/
inform_df.head()

[INFORM] available editions: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
[INFORM] edition 2026 - data years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
[INFORM] IDN (IDN) edition 2026, data year 2025: 78/124 indicators matched.
[INFORM] not matched (46): INFORM RISK, HAZARD & EXPOSURE, Natural, VULNERABILITY, LACK OF COPING CAPACITY, Lack of Reliability (*), NH.EQ.GEM.MMI6.SP, NH.EQ.GEM.MMI8.SP, HA.NAT.EQ.EXT-ABS, HA.NAT.EQ.EXT-REL, HA.NAT.EQ.INT-ABS, HA.NAT.EQ.INT-REL, Earthquake, POP, POP, EN.POP.DNST, Development & Deprivation, Inequality, FTS, FTS-INS, ODA-INS, Economic Dependency, Socio-Economic Vulnerability, VU.VG.UP.REF-TOT, VU.VG.UP.IDP-TOT, VU.VG.UP.RET-REF-TOT, Uprooted people, Children U5, VU.VGR.OG.FS.MA.ADSA, VU.VGR.OG.FS.MA.PU, Food Security, Recent Shocks, Health Conditions, Other Vulnerable Groups, Vulnerable Groups, DRR, Governance, Institutional, Communication, SH.STA.BASS.ZS, Physical infrastructure, Access to health care, Infrastructure, Proj

,source,time,time_type,spatial_level,spatial_resolution_m,crs,country,iso3,edition,data_year,...,category,indicator_id,indicator_name,type,value,unit,reference_year,matched,doubtful,doubt_reason
0,inform,2025,data_year,country,None,None,IDN,IDN,2026,2025,...,Composite scores (output finali INFORM),INFORM RISK,INFORM Risk (composite),aggregated,NaN,0-10,NaN,False,False,NaN
1,inform,2025,data_year,country,None,None,IDN,IDN,2026,2025,...,Composite scores (output finali INFORM),HAZARD & EXPOSURE,Hazard & Exposure (dimensione),aggregated,NaN,0-10,NaN,False,False,NaN
2,inform,2025,data_year,country,None,None,IDN,IDN,2026,2025,...,Composite scores (output finali INFORM),Natural,Natural hazard (categoria),aggregated,NaN,0-10,NaN,False,False,NaN
3,inform,2025,data_year,country,None,None,IDN,IDN,2026,2025,...,Composite scores (output finali INFORM),VULNERABILITY,Vulnerability (dimensione),aggregated,NaN,0-10,NaN,False,False,NaN
4,inform,2025,data_year,country,None,None,IDN,IDN,2026,2025,...,Composite scores (output finali INFORM),LACK OF COPING CAPACITY,Lack of Coping Capacity (dimensione),aggregated,NaN,0-10,NaN,False,False,NaN


9 DMT - IOM



In [16]:
from sources.iom.iom import download_dtm

download_dtm("AFG", "2025-04-01", "2026-01-01", admin_levels=0)

IOM DTM download | AFG | 2025-04-01 -> 2026-01-01 | admin [0]
  output folder: output\AFG\2025\country\iom
  ✓ connected to DTM API (v3)

  [1/1] admin0: requesting...
      ✓ 160 records received in 0.7s
      reporting dates   : 2025-05-31 -> 2025-05-31
      rounds            : 1
      admin units       : 1 distinct admin0Name
      IDP values        : 160 filled, 0 missing
      operation         : Climate and mobility in Afghanistan (160)
      displacementReason: Other reason (35), Natural disaster (35), Natural disaster; Other reason (33), Economic reasons (32), Natural disaster; Economic reasons (24) +1 more
      ✓ saved dtm_admin0_2025-04-01_2026-01-01.json, dtm_admin0_2025-04-01_2026-01-01.parquet, dtm_admin0_2025-04-01_2026-01-01.meta.json

------------------------------------------------------------
Done in 0.7s. OK: admin0=160 rows
Files in: C:\Users\decosiv\Desktop\codes\dataset_creation_pt2\output\AFG\2025\country\iom


{0:                                operation   admin0Name admin0Pcode  \
 0    Climate and mobility in Afghanistan  Afghanistan         AFG   
 1    Climate and mobility in Afghanistan  Afghanistan         AFG   
 2    Climate and mobility in Afghanistan  Afghanistan         AFG   
 3    Climate and mobility in Afghanistan  Afghanistan         AFG   
 4    Climate and mobility in Afghanistan  Afghanistan         AFG   
 ..                                   ...          ...         ...   
 155  Climate and mobility in Afghanistan  Afghanistan         AFG   
 156  Climate and mobility in Afghanistan  Afghanistan         AFG   
 157  Climate and mobility in Afghanistan  Afghanistan         AFG   
 158  Climate and mobility in Afghanistan  Afghanistan         AFG   
 159  Climate and mobility in Afghanistan  Afghanistan         AFG   
 
      numPresentIdpInd        reportingDate  yearReportingDate  \
 0                 848  2025-05-31T00:00:00               2025   
 1                 865 

In [19]:
import json


# Read and calculate
with open(r"C:\Users\decosiv\Desktop\codes\dataset_creation_pt2\output\AFG\2025\country\iom\dtm_admin0_2025-04-01_2026-01-01.json", "r", encoding="utf-8") as file:
    data = json.load(file)

total_idp = sum(item.get("numPresentIdpInd") or 0 for item in data)

print(f"Total Present IDPs: {total_idp}")

Total Present IDPs: 3109221


In [ ]:
from sources.hdx import (download_idps, download_humanitarian_needs, download_refugees,
                         download_returnees, download_food_security, download_food_prices,
                         download_poverty_rate)



download_idps("AFG", "2025-04-01", "2026-01-01", admin_level=0)


HDX HAPI | idps (IOM - DTM) | AFG | 2025-04-01 -> 2026-01-01 | admin 0
  output folder: output\AFG\2025\country\hdx\idps
  ✓ app identifier found

  [1/1] requesting affected-people/idps (location_code=AFG, admin_level=0)...
      ✓ 1 rows in 1.0s

  checks (1 rows):
      reference periods : 1
         2025-05-31 -> 2025-05-31  (1 rows)
      admin levels      : admin0=1
      population        : 1 filled, 0 missing, range 3.109e+06 .. 3.109e+06
      operation         : Climate and mobility in Afghanistan (1)
      assessment_type   : BA (1)
      reporting_round   : 2 (1)
      HDX resources     : 1 distinct resource_hdx_id

  ✓ saved idps_2025-04-01_2026-01-01__admin0.json  (+ .parquet, .meta.json)
------------------------------------------------------------
Done in 1.0s: 1 rows | window: 2025-04-01..2026-01-01


,location_code,location_name,admin1_code,admin1_name,admin2_code,admin2_name,admin_level,resource_hdx_id,reporting_round,assessment_type,operation,population,reference_period_start,reference_period_end
0,AFG,Afghanistan,None,None,None,None,0,80911e9b-7527-469a-a545-4074860e1288,2,BA,Climate and mobility in Afghanistan,3109221,2025-05-31T00:00:00,2025-05-31T23:59:59


In [24]:
download_refugees("AFG", "2025-04-01", "2026-01-01")

HDX HAPI | refugees (UNHCR) | AFG | 2025-04-01 -> 2026-01-01 | admin all
  output folder: output\AFG\2025\country\hdx\refugees
  ✓ app identifier found

  [1/2] requesting affected-people/refugees-persons-of-concern (asylum_location_code=AFG)...
      ✓ 117 rows in 7.8s

  [2/2] requesting affected-people/refugees-persons-of-concern (origin_location_code=AFG)...
      ✓ 2470 rows in 2.6s

  removed 26 duplicate rows (same row from different queries)

  checks (2561 rows):
      reference periods : 1
         2025-01-01 -> 2025-12-31  (2561 rows)
      population        : 2561 filled, 0 missing, range 0 .. 1.159e+06
      population_group  : REF (1248), ASY (1105), OOC (143), OIP (52), HST (13)
      origin_location_code: AFG (2470), IRN (26), PAK (26), BGD (13), IND (13), TJK (13)
      asylum_location_code: AFG (117), PAK (52), ALB (39), AUS (39), BIH (39), HRV (39) +105 more
      gender            : f (1182), m (1182), all (197)
      HDX resources     : 1 distinct resource_hdx_id



,resource_hdx_id,population_group,gender,age_range,min_age,max_age,population,reference_period_start,reference_period_end,origin_location_code,origin_location_name,asylum_location_code,asylum_location_name
0,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,OOC,f,0-4,0.0,4.0,26397,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
1,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,OOC,f,12-17,12.0,17.0,17117,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
2,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,OOC,f,18-59,18.0,59.0,63348,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
3,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,OOC,f,5-11,5.0,11.0,33645,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
4,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,OOC,f,60+,60.0,NaN,4416,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2556,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,ASY,m,18-59,18.0,59.0,0,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,ZMB,Zambia
2557,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,ASY,m,5-11,5.0,11.0,0,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,ZMB,Zambia
2558,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,ASY,m,60+,60.0,NaN,0,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,ZMB,Zambia
2559,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,ASY,m,all,NaN,NaN,7,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,ZMB,Zambia


In [25]:
download_returnees("AFG", "2025-04-01", "2026-01-01")

HDX HAPI | returnees (UNHCR) | AFG | 2025-04-01 -> 2026-01-01 | admin all
  output folder: output\AFG\2025\country\hdx\returnees
  ✓ app identifier found

  [1/2] requesting affected-people/returnees (asylum_location_code=AFG)...
      ✓ 26 rows in 1.0s

  [2/2] requesting affected-people/returnees (origin_location_code=AFG)...
      ✓ 26 rows in 0.1s

  removed 26 duplicate rows (same row from different queries)

  checks (26 rows):
      reference periods : 1
         2025-01-01 -> 2025-12-31  (26 rows)
      population        : 26 filled, 0 missing, range 153 .. 1.947e+06
      population_group  : RET (13), RDP (13)
      origin_location_code: AFG (26)
      asylum_location_code: AFG (26)
      gender            : f (12), m (12), all (2)
      HDX resources     : 1 distinct resource_hdx_id

  ✓ saved returnees_2025-04-01_2026-01-01.json  (+ .parquet, .meta.json)
------------------------------------------------------------
Done in 1.2s: 26 rows | window: 2025-04-01..2026-01-01


,resource_hdx_id,population_group,gender,age_range,min_age,max_age,population,reference_period_start,reference_period_end,origin_location_code,origin_location_name,asylum_location_code,asylum_location_name
0,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,f,0-4,0.0,4.0,127879,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
1,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,f,12-17,12.0,17.0,88109,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
2,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,f,18-59,18.0,59.0,303588,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
3,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,f,5-11,5.0,11.0,162881,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
4,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,f,60+,60.0,NaN,21384,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
5,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,f,all,NaN,NaN,703841,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
6,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,m,0-4,0.0,4.0,124426,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
7,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,m,12-17,12.0,17.0,96692,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
8,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,m,18-59,18.0,59.0,817014,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan
9,7e0b2b37-73b8-4c69-bb11-6ba954fa0cd9,RET,m,5-11,5.0,11.0,169006,2025-01-01T00:00:00,2025-12-31T23:59:59,AFG,Afghanistan,AFG,Afghanistan


In [26]:
download_humanitarian_needs("AFG", "2025-04-01", "2026-01-01")

HDX HAPI | humanitarian_needs (OCHA - HPC/JIAF) | AFG | 2025-04-01 -> 2026-01-01 | admin all
  output folder: output\AFG\2025\country\hdx\humanitarian_needs
  ✓ app identifier found

  [1/1] requesting affected-people/humanitarian-needs (location_code=AFG)...
      page 1: 10000 rows (total 10000)
      page 2: 10000 rows (total 20000)
      page 3: 10000 rows (total 30000)
      page 4: 10000 rows (total 40000)
      page 5: 10000 rows (total 50000)
      page 6: 10000 rows (total 60000)
      page 7: 10000 rows (total 70000)
      page 8: 10000 rows (total 80000)
      page 9: 10000 rows (total 90000)
      page 10: 10000 rows (total 100000)
      page 11: 10000 rows (total 110000)
      page 12: 10000 rows (total 120000)
      page 13: 10000 rows (total 130000)
      page 14: 10000 rows (total 140000)
      page 15: 10000 rows (total 150000)
      page 16: 10000 rows (total 160000)
      page 17: 10000 rows (total 170000)
      page 18: 10000 rows (total 180000)
      page 19: 10000

,location_code,location_name,admin1_code,admin1_name,admin2_code,admin2_name,admin_level,resource_hdx_id,sector_code,category,population_status,population,reference_period_start,reference_period_end,sector_name
0,AFG,Afghanistan,AF01,Kabul,AF0101,Kabul,2,22093993-e23b-45c8-b84f-61e4a414ebbb,FSC,Adults,INN,689573,2025-01-01T00:00:00,2025-12-08T23:59:59,Food Security
1,AFG,Afghanistan,AF01,Kabul,AF0101,Kabul,2,22093993-e23b-45c8-b84f-61e4a414ebbb,FSC,Adults,TGT,662335,2025-01-01T00:00:00,2025-12-08T23:59:59,Food Security
2,AFG,Afghanistan,AF01,Kabul,AF0101,Kabul,2,22093993-e23b-45c8-b84f-61e4a414ebbb,HEA,Adults,INN,296845,2025-01-01T00:00:00,2025-12-08T23:59:59,Health
3,AFG,Afghanistan,AF01,Kabul,AF0101,Kabul,2,22093993-e23b-45c8-b84f-61e4a414ebbb,HEA,Adults,TGT,151996,2025-01-01T00:00:00,2025-12-08T23:59:59,Health
4,AFG,Afghanistan,AF01,Kabul,AF0101,Kabul,2,22093993-e23b-45c8-b84f-61e4a414ebbb,Intersectoral,Adults,INN,1077150,2025-01-01T00:00:00,2025-12-08T23:59:59,Intersectoral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225724,AFG,Afghanistan,NaN,NaN,NaN,NaN,0,22093993-e23b-45c8-b84f-61e4a414ebbb,PRO-MIN,Women,TGT,276486,2025-01-01T00:00:00,2025-12-08T23:59:59,Mine Action
225725,AFG,Afghanistan,NaN,NaN,NaN,NaN,0,22093993-e23b-45c8-b84f-61e4a414ebbb,SHL,Women,INN,1198959,2025-01-01T00:00:00,2025-12-08T23:59:59,Emergency Shelter and NFI
225726,AFG,Afghanistan,NaN,NaN,NaN,NaN,0,22093993-e23b-45c8-b84f-61e4a414ebbb,SHL,Women,TGT,237385,2025-01-01T00:00:00,2025-12-08T23:59:59,Emergency Shelter and NFI
225727,AFG,Afghanistan,NaN,NaN,NaN,NaN,0,22093993-e23b-45c8-b84f-61e4a414ebbb,WSH,Women,INN,4901007,2025-01-01T00:00:00,2025-12-08T23:59:59,Water Sanitation Hygiene


In [47]:
download_food_prices("AFG", "2025-04-01", "2026-01-01")

HDX HAPI | food_prices (WFP) | AFG | 2025-04-01 -> 2026-01-01 | admin all
  output folder: output\AFG\2025\country\hdx\food_prices
  ✓ app identifier found

  [1/1] requesting food-security-nutrition-poverty/food-prices-market-monitor (location_code=AFG)...
      ✓ 4720 rows in 16.0s

  checks (4720 rows):
      reference periods : 10
         2025-03-15 -> 2025-04-14  (472 rows)
         2025-04-15 -> 2025-05-14  (472 rows)
         2025-05-15 -> 2025-06-14  (473 rows)
         2025-06-15 -> 2025-07-14  (472 rows)
         2025-07-15 -> 2025-08-14  (472 rows)
         2025-08-15 -> 2025-09-14  (472 rows)
         ... +4 more
      admin levels      : admin2=4720
      distinct admin2_name: 34
      distinct admin1_name: 34
      price             : 4720 filled, 0 missing, range 10 .. 1,000
      commodity_category: cereals and tubers (2010), non-food (1350), miscellaneous food (680), pulses and nuts (340), oil and fats (340)
      price_type        : Retail (4720)
      price_flag    

,location_code,location_name,admin1_code,admin1_name,admin2_code,admin2_name,admin_level,resource_hdx_id,market_code,market_name,...,commodity_category,currency_code,unit,price_flag,price_type,price,lat,lon,reference_period_start,reference_period_end
0,AFG,Afghanistan,AF10,Bamyan,AF1001,Bamyan,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,268,Bamyan,...,cereals and tubers,AFN,KG,aggregate,Retail,83.93,34.82,67.82,2025-03-15T00:00:00,2025-04-14T23:59:59
1,AFG,Afghanistan,AF10,Bamyan,AF1001,Bamyan,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,268,Bamyan,...,cereals and tubers,AFN,KG,aggregate,Retail,84.88,34.82,67.82,2025-04-15T00:00:00,2025-05-14T23:59:59
2,AFG,Afghanistan,AF10,Bamyan,AF1001,Bamyan,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,268,Bamyan,...,cereals and tubers,AFN,KG,aggregate,Retail,82.61,34.82,67.82,2025-05-15T00:00:00,2025-06-14T23:59:59
3,AFG,Afghanistan,AF10,Bamyan,AF1001,Bamyan,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,268,Bamyan,...,cereals and tubers,AFN,KG,aggregate,Retail,85.97,34.82,67.82,2025-06-15T00:00:00,2025-07-14T23:59:59
4,AFG,Afghanistan,AF10,Bamyan,AF1001,Bamyan,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,268,Bamyan,...,cereals and tubers,AFN,KG,aggregate,Retail,85.56,34.82,67.82,2025-07-15T00:00:00,2025-08-14T23:59:59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4715,AFG,Afghanistan,AF13,Paktya,AF1306,Sayed Karam,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,3695,Paktya,...,miscellaneous food,AFN,KG,aggregate,Retail,50.00,33.63,69.39,2025-08-15T00:00:00,2025-09-14T23:59:59
4716,AFG,Afghanistan,AF13,Paktya,AF1306,Sayed Karam,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,3695,Paktya,...,miscellaneous food,AFN,KG,aggregate,Retail,50.00,33.63,69.39,2025-09-15T00:00:00,2025-10-14T23:59:59
4717,AFG,Afghanistan,AF13,Paktya,AF1306,Sayed Karam,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,3695,Paktya,...,miscellaneous food,AFN,KG,aggregate,Retail,50.00,33.63,69.39,2025-10-15T00:00:00,2025-11-14T23:59:59
4718,AFG,Afghanistan,AF13,Paktya,AF1306,Sayed Karam,2,d62af4be-cff6-437b-89a3-67f8fa4c53bf,3695,Paktya,...,miscellaneous food,AFN,KG,aggregate,Retail,50.00,33.63,69.39,2025-11-15T00:00:00,2025-12-14T23:59:59


In [48]:
download_poverty_rate("AFG", "2025-04-01", "2026-01-01")

HDX HAPI | poverty_rate (OPHI - MPI) | AFG | 2025-04-01 -> 2026-01-01 | admin all
  output folder: output\AFG\2025\country\hdx\poverty_rate
  ✓ app identifier found

  [1/1] requesting food-security-nutrition-poverty/poverty-rate (location_code=AFG)...
      ✓ 0 rows in 1.1s

  ! nothing overlaps 2025-04-01 -> 2026-01-01; looking for the latest earlier release...
      ✓ using 35 rows from the release ending 2023-12-31

  checks (35 rows):
      reference periods : 1
         2022-01-01 -> 2023-12-31  (35 rows)
      admin levels      : admin0=1, admin1=34
      distinct admin1_name: 34
      mpi               : 35 filled, 0 missing, range 0.0682 .. 0.4437
      headcount_ratio   : 35 filled, 0 missing, range 15.14 .. 81.27
      intensity_of_deprivation: 35 filled, 0 missing, range 42.79 .. 56.47
      vulnerable_to_poverty: 35 filled, 0 missing, range 13.97 .. 45.77
      in_severe_poverty : 35 filled, 0 missing, range 4.006 .. 56.39
      HDX resources     : 1 distinct resource_hdx_

,location_code,location_name,admin1_code,admin1_name,admin_level,resource_hdx_id,mpi,headcount_ratio,intensity_of_deprivation,vulnerable_to_poverty,in_severe_poverty,reference_period_start,reference_period_end
0,AFG,Afghanistan,AF01,Kabul,1,7c131db3-a172-4280-b37a-85f678192fbe,0.0682,15.1416,45.0587,34.9170,4.0778,2022-01-01T00:00:00,2023-12-31T23:59:59
1,AFG,Afghanistan,AF02,Kapisa,1,7c131db3-a172-4280-b37a-85f678192fbe,0.2991,61.3466,48.7569,28.6038,27.0128,2022-01-01T00:00:00,2023-12-31T23:59:59
2,AFG,Afghanistan,AF03,Parwan,1,7c131db3-a172-4280-b37a-85f678192fbe,0.2179,43.4951,50.0928,32.6293,18.9266,2022-01-01T00:00:00,2023-12-31T23:59:59
3,AFG,Afghanistan,AF04,Maidan Wardak,1,7c131db3-a172-4280-b37a-85f678192fbe,0.2544,51.0114,49.8755,32.5402,22.5734,2022-01-01T00:00:00,2023-12-31T23:59:59
4,AFG,Afghanistan,AF05,Logar,1,7c131db3-a172-4280-b37a-85f678192fbe,0.2152,42.0406,51.1920,41.9146,18.6930,2022-01-01T00:00:00,2023-12-31T23:59:59
5,AFG,Afghanistan,AF06,Nangarhar,1,7c131db3-a172-4280-b37a-85f678192fbe,0.2613,53.3680,48.9617,26.6321,25.2242,2022-01-01T00:00:00,2023-12-31T23:59:59
6,AFG,Afghanistan,AF07,Laghman,1,7c131db3-a172-4280-b37a-85f678192fbe,0.2902,57.4106,50.5403,28.2632,26.6103,2022-01-01T00:00:00,2023-12-31T23:59:59
7,AFG,Afghanistan,AF08,Panjsher,1,7c131db3-a172-4280-b37a-85f678192fbe,0.1137,26.5773,42.7859,45.7710,4.0063,2022-01-01T00:00:00,2023-12-31T23:59:59
8,AFG,Afghanistan,AF09,Baghlan,1,7c131db3-a172-4280-b37a-85f678192fbe,0.3332,68.5503,48.6077,21.0723,30.0120,2022-01-01T00:00:00,2023-12-31T23:59:59
9,AFG,Afghanistan,AF10,Bamyan,1,7c131db3-a172-4280-b37a-85f678192fbe,0.1557,33.7161,46.1769,37.0825,10.2924,2022-01-01T00:00:00,2023-12-31T23:59:59


In [1]:
from sources.idmc.gidd import download_gidd


download_gidd(iso3="AFG", start=2025, end=2026)

IDMC GIDD download | AFG | 2025 -> 2026
  output folder: output\AFG\2025\country\idmc

  [1/3] figures (disaggregated): requesting...
      ✓ 12 figures received in 2.2s
      years             : 2025
      events            : 5
      Figure cause      : Conflict (7), Disaster (5)
      Figure category   : Internal Displacements (7), IDPs (5)
      ✓ saved gidd_figures_AFG_2025_2026.json, gidd_figures_AFG_2025_2026.parquet, gidd_figures_AFG_2025_2026.meta.json

  [2/3] events (disasters): requesting...
      ✓ 1 records received in 1.0s
      years             : 2025
      events            : 1 (new displacements 252,815)
      hazard types      : Mixed disasters (1)
      ✓ saved gidd_events_AFG_2025_2026.json, gidd_events_AFG_2025_2026.parquet, gidd_events_AFG_2025_2026.meta.json

  [3/3] country x year totals: requesting...
      ✓ 1 records received in 1.0s
      years             : 2025
      conflict_new_displacement   : 168,265
      disaster_new_displacement   : 252,815
      c

{'figures':         ID ISO3      Country Geographical region Figure cause  Year  \
 0   180464  AFG  Afghanistan          South Asia     Conflict  2025   
 1   180463  AFG  Afghanistan          South Asia     Conflict  2025   
 2   185778  AFG  Afghanistan          South Asia     Conflict  2025   
 3   185779  AFG  Afghanistan          South Asia     Conflict  2025   
 4   204251  AFG  Afghanistan          South Asia     Conflict  2025   
 5   204252  AFG  Afghanistan          South Asia     Conflict  2025   
 6   233862  AFG  Afghanistan          South Asia     Disaster  2025   
 7   233863  AFG  Afghanistan          South Asia     Disaster  2025   
 8   233914  AFG  Afghanistan          South Asia     Disaster  2025   
 9   233915  AFG  Afghanistan          South Asia     Disaster  2025   
 10  233916  AFG  Afghanistan          South Asia     Disaster  2025   
 11  235030  AFG  Afghanistan          South Asia     Conflict  2025   
 
            Figure category Figure unit  Reported f